# Aula 15 — Experiment Tracking

Este notebook executa o mesmo treinamento com MLflow e W&B. O objetivo é observar a diferença entre um resultado solto e um run com parâmetros, métricas, artefatos e linhagem.

In [ ]:
%pip -q install mlflow 'wandb>=0.20,<1' 'scikit-learn>=1.3,<2' 'pandas>=2,<3' 'matplotlib>=3.7,<4' 'joblib>=1.3,<2'

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

ACTIVITY = Path.cwd()
if not (ACTIVITY / 'src' / 'train.py').exists():
    candidates = list(Path('/content').glob('**/aula15/monitor/atividade'))
    if candidates:
        ACTIVITY = candidates[0]
os.chdir(ACTIVITY)
sys.path.insert(0, str(ACTIVITY))
print('Atividade:', ACTIVITY)
assert (ACTIVITY / 'src' / 'train.py').exists(), 'Abra o notebook a partir da pasta atividade.'

## 1. Preparar os dados

O Wine é distribuído pelo scikit-learn e não exige download externo. O digest será registrado nos runs.

In [ ]:
subprocess.run([sys.executable, '-m', 'src.prepare_data'], check=True)

## 2. Três configurações no MLflow

Cada chamada cria um run independente. Em uma máquina local, os runs ficam em `mlruns/`.

In [ ]:
configs = [
    ['--model', 'random_forest', '--n-estimators', '50', '--max-depth', '3'],
    ['--model', 'random_forest', '--n-estimators', '150', '--max-depth', '5'],
    ['--model', 'logistic_regression', '--c', '1.0'],
]
for index, config in enumerate(configs, start=1):
    command = [sys.executable, '-m', 'src.train', '--tracker', 'mlflow', '--seed', '42', '--run-name', f'notebook-mlflow-{index}', *config]
    subprocess.run(command, check=True)

In [ ]:
# Consulta textual simples dos melhores runs.
subprocess.run([sys.executable, '-m', 'src.compare_runs', '--tracking-uri', 'sqlite:///./.mlflow/mlflow.db', '--metric', 'macro_f1'], check=True)

## 3. O mesmo treino no W&B offline

O modo offline não exige API key. Para publicar em um projeto W&B, remova `--wandb-mode offline`, faça `wandb login` e use `--wandb-mode online`.

In [ ]:
subprocess.run([
    sys.executable, '-m', 'src.train', '--tracker', 'wandb',
    '--wandb-mode', 'offline', '--seed', '42',
    '--model', 'random_forest', '--n-estimators', '150', '--max-depth', '5',
    '--run-name', 'notebook-wandb-offline'
], check=True)

## 4. Discussão

Escolha um run e justifique: (a) métrica principal, (b) parâmetros, (c) seed e split, (d) digest do dataset, e (e) artefatos disponíveis.

DVC entra como a camada que identifica e distribui a versão dos dados; MLflow/W&B registram a execução que consumiu esses dados.